# Langchain: The basics

#### Developed By: Manaranjan Pradhan
#### www.manaranjanp.com

*This Jupyter notebook is confidential and proprietary to Manaranjan Pradhan. It is intended solely for authorized training purposes. Unauthorized distribution, sharing, or reproduction of this notebook or its contents is strictly prohibited. This material is for personal learning within the training program only and may not be used for commercial purposes or shared with others. Unauthorized use may result in disciplinary action or legal consequences. If you have received this notebook without authorization, please contact manaranjan@gmail.com immediately and delete all copies.*

This notebook uses the current **LangChain core** APIs (`langchain-core` + `langchain-groq`) and the **LCEL** (`prompt | llm | parser`) runnable pattern. The deprecated `LLMChain` and `langchain-classic` imports have been removed.

In [1]:
!pip -q install langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.0 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

In [3]:
#os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


## Zero Shot Prompting


In [4]:
from langchain_groq import ChatGroq

In [5]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=256,
    max_retries=2,
)

In [6]:
text = """Classify the sentiment of the customer review below as Positive or Negative.
Respond with only the label.

Review: The seafood platter was fresh and flavorful. Loved the chic decor and the ambiance of the place.
Sentiment:"""

print(llm.invoke(text).content)

Positive


## Classifying List of Reviews

In [7]:
reviews = [
    "The seafood platter was fresh and flavorful. Loved the chic decor and the ambiance of the place.",
    "Impressive service! The staff was attentive and made excellent recommendations. Thoroughly enjoyed the evening.",
    "The restaurant's interior was a visual treat, beautifully paired with their gourmet dishes. A delightful experience!",
    "The food tasted alright, but the tables were not very clean, which was off-putting.",
    "Waited 30 minutes even with a reservation, and the main course was served cold. Disappointing visit."
]

## Using Prompt Templates

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [9]:
sentiment_template = """Classify the sentiment of the customer review below as Positive or Negative.
Respond with only the label.

Review: {review_text}
Sentiment:"""

# from_template infers the input variables ({review_text}) automatically.
review_prompt = PromptTemplate.from_template(sentiment_template)

### Generating Prompt with Templates

In [10]:
print(review_prompt.format(review_text=reviews[0]))

Classify the sentiment of the customer review below as Positive or Negative.
Respond with only the label.

Review: The seafood platter was fresh and flavorful. Loved the chic decor and the ambiance of the place.
Sentiment:


### Calling LLM with the prompt template

In [11]:
# LCEL: pipe the prompt into the model, then parse the message down to a plain string.
sentiment_chain = review_prompt | llm | StrOutputParser()

In [12]:
response = sentiment_chain.invoke({"review_text": reviews[0]})

response  # StrOutputParser returns a clean string

'Positive'

In [13]:
print("================================")

for review in reviews:
    print(review_prompt.format(review_text=review))
    print(sentiment_chain.invoke({"review_text": review}))
    print("================================")

Classify the sentiment of the customer review below as Positive or Negative.
Respond with only the label.

Review: The seafood platter was fresh and flavorful. Loved the chic decor and the ambiance of the place.
Sentiment:
Positive
Classify the sentiment of the customer review below as Positive or Negative.
Respond with only the label.

Review: Impressive service! The staff was attentive and made excellent recommendations. Thoroughly enjoyed the evening.
Sentiment:
Positive
Classify the sentiment of the customer review below as Positive or Negative.
Respond with only the label.

Review: The restaurant's interior was a visual treat, beautifully paired with their gourmet dishes. A delightful experience!
Sentiment:
Positive
Classify the sentiment of the customer review below as Positive or Negative.
Respond with only the label.

Review: The food tasted alright, but the tables were not very clean, which was off-putting.
Sentiment:
Negative
Classify the sentiment of the customer review belo

## Classifying Categories

In [14]:
category_template = """Classify the customer review into exactly one category:
Food Quality, Overall Hygiene, Restaurant Ambience, or Customer Service.
Respond with only the category name.

Review: The grilled chicken was seasoned to perfection and simply melted in the mouth.
Category: Food Quality

Review: Despite the crowd, the place was immaculately clean and the restrooms were spotless.
Category: Overall Hygiene

Review: The dim lighting and soothing jazz music provided an intimate and romantic setting.
Category: Restaurant Ambience

Review: We were kept waiting even after a confirmed reservation and the staff seemed disinterested.
Category: Customer Service

Review: {review_text}
Category:"""

category_prompt = PromptTemplate.from_template(category_template)

In [15]:
print(category_prompt.format(review_text=reviews[0]))

Classify the customer review into exactly one category:
Food Quality, Overall Hygiene, Restaurant Ambience, or Customer Service.
Respond with only the category name.

Review: The grilled chicken was seasoned to perfection and simply melted in the mouth.
Category: Food Quality

Review: Despite the crowd, the place was immaculately clean and the restrooms were spotless.
Category: Overall Hygiene

Review: The dim lighting and soothing jazz music provided an intimate and romantic setting.
Category: Restaurant Ambience

Review: We were kept waiting even after a confirmed reservation and the staff seemed disinterested.
Category: Customer Service

Review: The seafood platter was fresh and flavorful. Loved the chic decor and the ambiance of the place.
Category:


In [16]:
category_chain = category_prompt | llm | StrOutputParser()

In [17]:
response = category_chain.invoke({"review_text": reviews[0]})

response

'Food Quality'

In [18]:
for review in reviews:
    print(review)
    print(category_chain.invoke({"review_text": review}))
    print("================================")

The seafood platter was fresh and flavorful. Loved the chic decor and the ambiance of the place.
Food Quality
Impressive service! The staff was attentive and made excellent recommendations. Thoroughly enjoyed the evening.
Customer Service
The restaurant's interior was a visual treat, beautifully paired with their gourmet dishes. A delightful experience!
Restaurant Ambience
The food tasted alright, but the tables were not very clean, which was off-putting.
Overall Hygiene
Waited 30 minutes even with a reservation, and the main course was served cold. Disappointing visit.
Customer Service


## Chaining Multiple Prompts

In [19]:
response_to_customer = """Write a concise, professional response to a customer whose review expressed {sentiment} sentiment about {category}.

Appreciate positive feedback or address the specific concern, aiming to keep the customer engaged.
The response must be no more than 50 words."""

In [20]:
response_prompt = PromptTemplate.from_template(response_to_customer)

In [21]:
final_response_chain = response_prompt | llm | StrOutputParser()

In [22]:
# The two upstream chains feed their string outputs into {sentiment} and {category}.
complete_chain = (
    {
        "sentiment": sentiment_chain,
        "category": category_chain,
    }
    | final_response_chain
)

In [23]:
reviews[3]

'The food tasted alright, but the tables were not very clean, which was off-putting.'

In [24]:
response = complete_chain.invoke({"review_text": reviews[3]})

In [25]:
print(response)

"Sorry to hear about your hygiene concerns. We take this seriously and will review our protocols to ensure improvement."
